In [1]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import multiprocessing
import torch
import torch.nn as nn
import torch.optim as optim 
from torch.utils.data import ConcatDataset,DataLoader, Dataset, Subset 
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.transforms import v2
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torchinfo import summary
from tqdm.auto import tqdm
from PIL import Image
import random
import time
import copy

In [2]:
# Set SEED for reproducibility

SEED = 24520152

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
# Check accelerator

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(f'Using {device} device')

Using cuda device


In [4]:
# Train directory path

TRAIN_DIR = '/kaggle/input/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'
TRAIN_DIR

'/kaggle/input/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'

In [5]:
# Model weights
INPUT_DIR= '/kaggle/input/vgg19-figsharett-92'
INPUT_DIR

'/kaggle/input/vgg19-figsharett-92'

In [6]:
SAVE_DIR = '/kaggle/working/'
SAVE_DIR

'/kaggle/working/'

In [7]:
# 3 labels: Glioma Tumor, Meningioma Tumor, Pituitary Tumor
CLASS_NAMES = sorted([d for d in os.listdir(os.path.join(TRAIN_DIR, 'Subset_1')) if os.path.isdir(os.path.join(TRAIN_DIR, 'Subset_1', d))])
CLASS_NAMES

['Glioma Tumor', 'Meningioma Tumor', 'Pituitary Tumor']

In [8]:
# Model name

MODEL_NAME = 'VGG19'
MODEL_NAME

'VGG19'

In [9]:
# Hyperparameters

BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 3
DROPOUT_RATE = 0.3
SPLIT_RATIO = 0.8

In [10]:
def get_data_loaders(train_dir: str = TRAIN_DIR,batch_size: int = BATCH_SIZE, split_ratio: float = SPLIT_RATIO) -> tuple[DataLoader, DataLoader]:
    """
    Constructs PyTorch DataLoaders with a strict training/validation split.

    This function implements the 'Two-Lens' strategy:
    1. It loads the entire dataset twice: once with Augmentation policies (Train View) 
       and once with only Normalization policies (Validation View).
    2. It shuffles indices and splits them (e.g., 80/20).
    3. It maps the training indices to the 'Train View' and validation indices 
       to the 'Validation View'.
    
    This ensures that validation data is NEVER augmented (preserving ground truth),
    while training data receives stochastic transformations for regularization.

    Args:
        train_dir (str): Root directory containing 'Subset_X' folders.
        batch_size (int): Number of samples per batch.
        split_ratio (float): Proportion of data to use for training (0.0 to 1.0).

    Returns:
        Tuple[DataLoader, DataLoader]: A tuple containing (train_loader, val_loader).
    """
    
    # Standard ImageNet normalization statistics 
    norm_mean=[0.485, 0.456, 0.406]
    norm_std=[0.229, 0.224, 0.225]

    # Training Transform Pipeline
    train_transform = v2.Compose([
        # Resize to 256x256 first. This provides a buffer for subsequent 
        # rotation/translation and cropping, preventing black border artifacts.
        v2.Resize(size=256),

        # Apply Data Augmentation
        v2.RandomHorizontalFlip(),
        v2.RandomRotation(degrees=36),
        v2.RandomAffine(degrees=0, scale=(0.9, 1.1)),
        v2.ColorJitter(brightness=0.1, contrast=0.1),

        # Use CenterCrop to focus on the primary subject
        v2.CenterCrop(size=224),

        # Convert PIL/Numpy to Tensor, cast to Float32, and rescale to [0, 1]
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),

        # Normalize using ImageNet mean and std
        v2.Normalize(mean=norm_mean, std=norm_std),
    ])

    # Validation Transform Pipeline
    val_transform = v2.Compose([
        v2.Resize(size=256),
        v2.CenterCrop(size=224),
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),
        v2.Normalize(mean=norm_mean, std=norm_std)
    ])

    # Worker Configuration
    # Determine the optimal number of CPU workers to prevent bottlenecks.
    # Capped at 4 to avoid excessive memory overhead.
    num_workers = min(4, os.cpu_count())
    
    # We create lists to hold the sub-datasets.
    # Note: images are lazy-loaded.
    train_view_subsets = [] 
    val_view_subsets = []

    # List to store labels for stratification
    all_labels = [] 
    # Iterate through all subsets (1 to 5) to merge them into a single pool  
    for i in range(1, 6):
        subset_path = os.path.join(train_dir, f'Subset_{i}')
        if os.path.exists(subset_path):
            # Load dataset reference
            ds_train_view = ImageFolder(root=subset_path, transform=train_transform)
            ds_val_view = ImageFolder(root=subset_path, transform=val_transform)
            
            train_view_subsets.append(ds_train_view)
            val_view_subsets.append(ds_val_view)
            
            # Extract labels for Stratification
            # ImageFolder.targets gives a list of labels for that folder
            all_labels.extend(ds_train_view.targets)
        else:
            print(f"Path not found: {subset_path}")

    # Concatenate all subsets into two massive datasets
    combined_train_view = ConcatDataset(train_view_subsets)
    combined_val_view = ConcatDataset(val_view_subsets)
    # We split indices based on 'all_labels' to ensure equal class distribution
    total_samples = len(combined_train_view)
    indices = list(range(total_samples))

    train_indices, val_indices = train_test_split(indices, train_size=split_ratio, shuffle=True, random_state=SEED, stratify= all_labels)
    train_dataset = Subset(combined_train_view, train_indices)
    val_dataset = Subset(combined_val_view, val_indices)
    

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader

In [11]:
class VGG19(nn.Module):
    """
    VGG19-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: VGG19 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> None:
        super().__init__()

        # Load Pre-trained VGG19
        weights = models.VGG19_Weights.IMAGENET1K_V1
        backbone = models.vgg19(weights=weights)

        # VGG19 .features contains all Conv/Relu/MaxPool layers
        self.features = backbone.features

        # Freezing parameters to prevent updating during training
        for param in self.features.parameters():
            param.requires_grad = False

        # Define Custom Classifier Head
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 512, H, W) -> (Batch, 512, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 512, 1, 1) -> (Batch, 512)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=512, out_features=num_classes) # VGG19 features output exactly 512 channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W)

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """

        # Feature extraction (Frozen)
        x = self.features(x)

        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)

        # Output Logits
        logits = self.classifier(x)

        return logits

In [12]:
def build_vgg19(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> VGG19:
    """
    Factory function to instantiate the customized VGG19 model for Transfer Learning.

    This function initializes a `VGG19` which includes:
    1. A frozen VGG19 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        VGG19Classifier: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """

    model = VGG19(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [13]:
def get_model(i: int, input_dir: str = INPUT_DIR)-> nn.Module:
    """
    Loads the trained VGG19 model weights for a specific block.

    Args:
        i (int): Block index (e.g., 6, 5, 4...)..
        input_dir (str): Directory containing .pth files.
    Returns:
        nn.Module: The model with loaded weights set to eval mode, or None if file missing.
    """
   
    # Construct model
    model = build_vgg19().to(device)
    
    # Make file path
    # f'{model_name}_block_{block}.pth'
    # Example: /kaggle/input/vgg19-mri-ben-32/VGG19_block_1.pth
    filename = f'VGG19_block_{i}.pth'
    filepath = os.path.join(input_dir, filename)
    
    # Load weights
    if os.path.exists(filepath):
        # Load state_dict 
        state_dict = torch.load(filepath, map_location=device, weights_only = True)
        model.load_state_dict(state_dict)
        print(f"Loaded: {filename}")
    else:
        print(f"Warning: File {filename} not found at {input_dir}")
        return None

    # Evaluation mode
    model.to(device)
    model.eval() 
    
    return model

In [14]:
def calculate_macro_specificity(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int = NUM_CLASSES) -> float:
    """
    Computes the Macro-Average Specificity (True Negative Rate) for multi-class classification.

    This function calculates specificity using the "One-vs-Rest" strategy:
    1. For each class, it treats that class as "Positive" and all other classes as "Negative".
    2. It computes True Negatives (TN) and False Positives (FP) for that specific class.
    3. It calculates specificity for that class using the formula: Specificity = TN / (TN + FP).
    4. Finally, it returns the unweighted mean (macro-average) of specificity scores across all classes.

    Args:
        y_true (np.ndarray): 1D array containing the ground truth class labels. 
            Shape: (n_samples,). Example: [0, 1, 2, 0]
        y_pred (np.ndarray): 1D array containing the predicted class labels (not probabilities). 
            Shape: (n_samples,). Example: [0, 2, 2, 0]
        num_classes (int, optional): The total number of unique classes in the dataset. 
            Defaults to 3.

    Returns:
        float: The macro-averaged specificity score. 
               Range is [0.0, 1.0], where 1.0 indicates perfect identification of negative cases.

    Note:
        A small epsilon (1e-8) is added to the denominator to prevent ZeroDivisionError 
        in cases where (TN + FP) equals 0.
    """
    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    
    # Initialize list to store specificity for each class
    specs = []

    for i in range(num_classes):
        # True Positives
        TP = cm[i, i]
        # False Negatives:
        FN = cm[i, :].sum() - TP
        # False Positives
        FP = cm[:, i].sum() - TP
        # True Negatives
        TN = cm.sum() - (TP + FN + FP)
        # Specificity
        specificity = TN / (TN + FP + 1e-8)
        specs.append(specificity)

    # Return the mean to get a single Macro Average score
    return np.array(specs)

In [15]:
def evaluate_model(block_idx: int, input_dir: str) -> list:
    """
    Evaluates the model performance acrossblocks and
    computes evaluation metrics.
    Args:
        block_idx: index
        input_dir: Path to saved models.
    Returns:
        np.ndarray: A matrix of metrics for all blocks.
                    Shape: (num_blocks, 5_metrics).
                    Columns order: [Recall, Specificity, Precision, F1-Score, Accuracy].
    """
    # 1. Load Model
    model = get_model(block_idx, input_dir)
    if model is None:
        return [0, 0, 0, 0, 0] 

   
    _,valid_loader = get_data_loaders(TRAIN_DIR, BATCH_SIZE)

    y_true = []
    y_pred_probs = []

    # 3. Inference Loop
    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            
           
            probs = torch.softmax(outputs, dim=1)
            
            y_true.extend(labels.numpy())
            y_pred_probs.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_pred_probs = np.array(y_pred_probs)
    
    # 4. Metrics
    y_pred_label = np.argmax(y_pred_probs, axis=1)

    acc = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred_label, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred_label, average='macro', zero_division=0)
    spec_array = calculate_macro_specificity(y_true, y_pred_label, num_classes=NUM_CLASSES)
    specificity = np.mean(spec_array) # Take Mean to get one single score

    # Order: Recall, Specificity, Precision, F1, Accuracy
    return [recall, specificity, precision, f1, acc]

In [16]:
num_model = 6
metric_names = ['Recall', 'Specificity', 'Precision', 'F1-Score', 'Accuracy']

avg_metrics = {}

for m in range(num_model):
    block_idx = m + 1
    
    metrics_row = evaluate_model(block_idx, INPUT_DIR)
    avg_metrics[m] = {}
    for idx, metric_name in enumerate(metric_names):
        # Take metrics of all at model(m), at column metric (idx)
        value = metrics_row[idx]
        avg_metrics[m][metric_name] = value * 100

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 220MB/s]


Loaded: VGG19_block_1.pth
Loaded: VGG19_block_2.pth
Loaded: VGG19_block_3.pth
Loaded: VGG19_block_4.pth
Loaded: VGG19_block_5.pth
Loaded: VGG19_block_6.pth


In [17]:
row_names = ['FT: B$_1$-B$_6$', 'FT: B$_2$-B$_6$', 'FT: B$_3$-B$_6$', 'FT: B$_4$-B$_6$', 'FT: B$_5$-B$_6$', 'FT: B$_6$']
column_names = ['Recall', 'Specificity', 'Precision', 'F1-Score', 'Accuracy']

df_result = pd.DataFrame(avg_metrics).T
df_result.index = row_names
df_result.index.name = 'Fine-tuning'
df_result = df_result[column_names]
df_result = df_result.round(2)
df_result = df_result.reset_index()
df_result.style.hide(axis='index').format(precision=2)

Fine-tuning,Recall,Specificity,Precision,F1-Score,Accuracy
FT: B$_1$-B$_6$,95.41,98.07,96.35,95.84,96.41
FT: B$_2$-B$_6$,96.12,98.27,96.61,96.35,96.74
FT: B$_3$-B$_6$,96.06,98.18,96.39,96.22,96.57
FT: B$_4$-B$_6$,94.42,97.62,95.66,94.93,95.60
FT: B$_5$-B$_6$,90.18,95.70,91.73,90.81,92.01
FT: B$_6$,78.26,89.94,80.90,79.28,81.57


In [18]:
latex_table = df_result.to_latex(
    multicolumn=True,
    multirow=True,
    float_format="%.2f",
    label="tab:sens_spec"
)

print(latex_table)

\begin{table}
\label{tab:sens_spec}
\begin{tabular}{llrrrrr}
\toprule
 & Fine-tuning & Recall & Specificity & Precision & F1-Score & Accuracy \\
\midrule
0 & FT: B$_1$-B$_6$ & 95.41 & 98.07 & 96.35 & 95.84 & 96.41 \\
1 & FT: B$_2$-B$_6$ & 96.12 & 98.27 & 96.61 & 96.35 & 96.74 \\
2 & FT: B$_3$-B$_6$ & 96.06 & 98.18 & 96.39 & 96.22 & 96.57 \\
3 & FT: B$_4$-B$_6$ & 94.42 & 97.62 & 95.66 & 94.93 & 95.60 \\
4 & FT: B$_5$-B$_6$ & 90.18 & 95.70 & 91.73 & 90.81 & 92.01 \\
5 & FT: B$_6$ & 78.26 & 89.94 & 80.90 & 79.28 & 81.57 \\
\bottomrule
\end{tabular}
\end{table}

